# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane check:** staying on Lane 1 (Ranking Signal Analysis) — confirmed for this week.


## 0. Check two signals first

*Two signals my rule idea leans on. At least one backs a real FlyRank flag from the session.*

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Same decline label as ML-03: an OBSERVED outcome, not a rule I borrowed
eligible = df[df["clicks_prev_30d"] >= 5].copy()
eligible["declined"] = (eligible["clicks_last_30d"] < eligible["clicks_prev_30d"]).astype(int)

print("=" * 70)
print("SIGNAL 1: staleness -> declining  (behind FlyRank's refresh flags)")
print("Claim: pages that haven't been updated in a long time are more likely declining.")
print("=" * 70)
g1 = eligible.groupby("freshness_tier")["declined"].agg(["mean", "count"]).rename(
    columns={"mean": "decline_rate", "count": "n"})
print(g1)
print()
print("Verdict: FALSE. The two well-populated buckets (0-30 days, n=2,936; 91-180 days,")
print("n=2,063) show almost IDENTICAL decline rates (63.5% vs 62.3%) -- staleness isn't")
print("separating decliners from non-decliners here. The buckets that LOOK dramatic")
print("(31-90: 33.3%, 181+: 100%) both fall under the 50-row floor (n=12, n=8) -- noise")
print("wearing a costume, not a real pattern. This negative just saved the rule from")
print("leaning on a belief the data doesn't support.")
print()

print("=" * 70)
print("SIGNAL 2: position tier -> CTR  (behind FlyRank's CTR-fix logic)")
print("Claim: pages further from the top position get a lower click-through rate.")
print("=" * 70)
valid_pos = df[df["avg_position"] > 0]
g2 = valid_pos.groupby("position_tier")["ctr"].agg(["mean", "count"]).rename(
    columns={"mean": "mean_ctr", "count": "n"}).sort_values("mean_ctr", ascending=False)
print(g2)
print()
print("Verdict: CONFIRMED. Clear, monotonic drop in mean CTR as position tier worsens")
print("(2.76% at top_3, down to 0.15% at deep), every bucket well above the 50-row floor")
print("(smallest is n=1,116). This is the real signal behind the CTR-fix belief -- safe")
print("to build the rule around.")


SIGNAL 1: staleness -> declining  (behind FlyRank's refresh flags)
Claim: pages that haven't been updated in a long time are more likely declining.
                decline_rate     n
freshness_tier                    
0-30                0.634877  2936
181+                1.000000     8
31-90               0.333333    12
91-180              0.622879  2063

Verdict: FALSE. The two well-populated buckets (0-30 days, n=2,936; 91-180 days,
n=2,063) show almost IDENTICAL decline rates (63.5% vs 62.3%) -- staleness isn't
separating decliners from non-decliners here. The buckets that LOOK dramatic
(31-90: 33.3%, 181+: 100%) both fall under the 50-row floor (n=12, n=8) -- noise
wearing a costume, not a real pattern. This negative just saved the rule from
leaning on a belief the data doesn't support.

SIGNAL 2: position tier -> CTR  (behind FlyRank's CTR-fix logic)
Claim: pages further from the top position get a lower click-through rate.
               mean_ctr      n
position_tier            

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
rule_in_plain_words = """
A page deserves review first if it currently gets real search visibility (impressions)
AND its click-through rate is meaningfully below what other pages in its OWN position
tier typically get. That gap -- underperforming for your position -- is the confirmed
signal (Signal 2 above). Staleness is deliberately NOT part of this rule: Signal 1 showed
it doesn't reliably separate decliners from non-decliners in this data, so leaning on it
would be building on a belief, not evidence.

Reason codes this rule can output (ONE per page, priority order):
  1. underperforming_ctr_for_position -- big visibility, CTR far below tier median
  2. high_visibility_review           -- very high visibility, but no confirmed CTR gap
  3. monitor                         -- neither condition strongly triggered
"""
print(rule_in_plain_words)



A page deserves review first if it currently gets real search visibility (impressions)
AND its click-through rate is meaningfully below what other pages in its OWN position
tier typically get. That gap -- underperforming for your position -- is the confirmed
signal (Signal 2 above). Staleness is deliberately NOT part of this rule: Signal 1 showed
it doesn't reliably separate decliners from non-decliners in this data, so leaning on it
would be building on a belief, not evidence.

Reason codes this rule can output (ONE per page, priority order):
  1. underperforming_ctr_for_position -- big visibility, CTR far below tier median
  2. high_visibility_review           -- very high visibility, but no confirmed CTR gap
  3. monitor                         -- neither condition strongly triggered



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import numpy as np

def pct_rank(s):
    return s.rank(pct=True)

# Only score pages with a real position and real visibility -- can't rank what
# has no rankable position or no traffic at all
scoreable = df[(df["avg_position"] > 0) & (df["impressions_90d"] > 0)].copy()
print("scoreable rows:", len(scoreable))

# CTR gap vs the page's OWN position tier -- the confirmed signal, not a raw CTR number
tier_median_ctr = scoreable.groupby("position_tier")["ctr"].median()
scoreable["tier_median_ctr"] = scoreable["position_tier"].map(tier_median_ctr)
scoreable["ctr_gap"] = (scoreable["tier_median_ctr"] - scoreable["ctr"]).clip(lower=0)

scoreable["visibility_score"] = pct_rank(np.log1p(scoreable["impressions_90d"]))
scoreable["ctr_gap_score"] = pct_rank(scoreable["ctr_gap"])

# Transparent score: no fitted weights, just two readable percentile ranks averaged
scoreable["baseline_action_score"] = (
    0.5 * scoreable["visibility_score"] + 0.5 * scoreable["ctr_gap_score"]
).round(4)

def reason_code(row):
    if row["ctr_gap"] > 0 and row["ctr_gap_score"] >= 0.75 and row["visibility_score"] >= 0.5:
        return "underperforming_ctr_for_position"
    elif row["visibility_score"] >= 0.9:
        return "high_visibility_review"
    else:
        return "monitor"

scoreable["reason_code"] = scoreable.apply(reason_code, axis=1)

action_map = {
    "underperforming_ctr_for_position": "review_ctr",
    "high_visibility_review": "monitor_visibility",
    "monitor": "monitor",
}
scoreable["action"] = scoreable["reason_code"].map(action_map)

scoreable["rank"] = scoreable["baseline_action_score"].rank(method="first", ascending=False).astype(int)
scoreable = scoreable.sort_values("rank")

print()
print(scoreable["reason_code"].value_counts())

import os
os.makedirs("../outputs", exist_ok=True)
output_cols = ["content_id", "client_id", "rank", "baseline_action_score", "reason_code",
               "action", "position_tier", "ctr", "tier_median_ctr", "impressions_90d",
               "clicks_prev_30d", "clicks_last_30d", "word_count", "content_type"]
scoreable[output_cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print()
print("Wrote", len(scoreable), "rows to work/outputs/baseline_action_score.csv")


scoreable rows: 28795

reason_code
monitor                             25020
high_visibility_review               2788
underperforming_ctr_for_position      987
Name: count, dtype: int64

Wrote 28795 rows to work/outputs/baseline_action_score.csv


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top10 = scoreable[output_cols].head(10)
print(top10.to_string(index=False))

top10_review = """
TOP-10 REVIEW (action | why it's there | what would make it wrong):

1. review_ctr | page_1, 208,678 impressions, CTR 0.00% vs tier median 0.16% | Zero clicks
   despite huge visibility could also mean the page isn't actually indexed/clickable, not
   a title/snippet problem -- worth confirming it's live before "fixing" copy.
2. review_ctr | page_1, 22,456 impressions, CTR 0.00%, 3,803 words | Could be a page that
   fully answers the query in the snippet itself (a "position-zero" effect) -- zero clicks
   might be expected here, not a bug.
3. review_ctr | page_1, 140,079 impressions, CTR 0.01%, clicks_prev_30d=4 | Below my own
   >=5 reliability floor for click history -- the CTR gap is real, but I can't confirm
   any trend on this specific page from clicks alone.
4. review_ctr | page_1, 223,271 impressions, clicks 19->36 (UP, not down) | This page's
   clicks are already improving -- flagging it for a CTR fix may be redundant or could
   undo something that's already working; check recent history before touching it.
5. review_ctr | page_1, 112,434 impressions, clicks_prev_30d=6, clicks fell to 1 | Strongest
   case in the top 10: visibility, confirmed CTR gap, and an observed decline all agree.
   Still, a 6-to-1 swing is only 5 raw clicks -- easy to overstate at this scale.
6. review_ctr | page_1, 119,217 impressions, clicks 10->3 | Same pattern as #5, one tier
   down in raw count -- could be a seasonal dip in this specific keyword, not a lasting one.
7. review_ctr | page_1, 16,786 impressions, clicks_prev_30d=0 | No click history at all to
   cross-check against -- this pick rests entirely on the CTR-gap belief, with nothing to
   confirm or deny it.
8. review_ctr | page_1, 134,055 impressions, clicks 19->13, word_count missing | Missing
   word_count is a small red flag -- worth confirming this is a real, complete page before
   spending review time on it.
9. review_ctr | page_1, 123,469 impressions, clicks 11->9, word_count missing | Same missing-
   word_count concern as #8, plus an 11-to-9 dip is a very small swing to call "declining."
10. review_ctr | page_1, 16,156 impressions, clicks_prev_30d=0, 3,343 words | Same as #7 --
    no click history at all, resting entirely on the CTR-gap belief alone.
"""
print(top10_review)


          content_id         client_id  rank  baseline_action_score                      reason_code     action position_tier  ctr  tier_median_ctr  impressions_90d  clicks_prev_30d  clicks_last_30d  word_count    content_type
content_c8e9d6ab9013 client_19581e27de     1                 0.9645 underperforming_ctr_for_position review_ctr        page_1 0.00             0.16           208678                0                0         NaN keyword article
content_f986bd514b6e client_7f2253d7e2     2                 0.9382 underperforming_ctr_for_position review_ctr        page_1 0.00             0.16            22456                0                0      3803.0 keyword article
content_453722754fea client_f369cb89fc     3                 0.9284 underperforming_ctr_for_position review_ctr        page_1 0.01             0.16           140079                4                3      2700.0 keyword article
content_c84a0ab98e90 client_f369cb89fc     4                 0.9278 underperforming_ctr_for_

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
weak_picks_and_leakage = """
WEAK PICKS (found by looking hard, per the skill's own instruction):
- #4: clicks are RISING (19 -> 36), not declining -- the rule flagged it purely on the
  static CTR-gap number and missed the page's own recent direction. A rule this simple
  can't see trend, only snapshot -- a real weakness worth naming, not hiding.
- #3, #7, #10: clicks_prev_30d is 4, 0, and 0 -- all below my own reliability floor
  (>=5) from ML-03. These three picks rest entirely on the CTR-gap belief with no
  click-based cross-check at all.

LEAKAGE CHECK:
- No FlyRank product flags used anywhere (health_score, priority_score, action_type,
  refresh_tier are not in this dataset at all, and were never referenced).
- Honest caveat: ctr, avg_position, and impressions_90d are 90-day aggregates that
  PARTIALLY OVERLAP the same last_30d window used to build my declined label (from
  ML-03). That means precision@10 below is a same-period consistency check -- does the
  rule agree with what already happened -- NOT a forecast of future decline. I'm not
  claiming this rule predicts anything; it prioritizes today's review queue using
  today's signals.
- Only 5 of the top 10 even have a computable declined label (clicks_prev_30d >= 5
  for the rest). n=5 is far below the 50-row floor -- I can report the number, but I
  will not call it a reliable precision estimate.
"""
print(weak_picks_and_leakage)

# Report the number anyway, with n printed, so the reader can judge for themselves
eligible_labels = eligible[["content_id", "declined"]]
merged = scoreable.merge(eligible_labels, on="content_id", how="left")
merged = merged.sort_values("rank")
k = 10
topk = merged.head(k)
base_rate = eligible["declined"].mean()
labeled_n = topk["declined"].notna().sum()
precision_at_10 = topk["declined"].mean()

print(f"Base rate of declined (all eligible rows, n={len(eligible)}): {base_rate:.1%}")
print(f"Precision@10 (n with a computable label: {labeled_n}/10): {precision_at_10:.1%}")
print("-> Directional only. n=5 is below the 50-row floor -- not a claim, a data point.")


# Write the run's "receipt" -- small enough to commit, unlike the full CSV
import json as _json

metrics = {
    "lane": "Lane 1: Ranking Signal Analysis",
    "signal_verdicts": {
        "staleness_vs_decline": {
            "verdict": "FALSE",
            "linked_flag": "refresh flags",
            "buckets_n": {"0-30": 2936, "91-180": 2063, "31-90": 12, "181+": 8},
        },
        "position_tier_vs_ctr": {
            "verdict": "CONFIRMED",
            "linked_flag": "CTR-fix logic",
            "buckets_n": {"top_3": 1116, "page_1": 11814, "striking": 7304, "page_3_5": 7242, "deep": 1319},
        },
    },
    "rule": "underperforming_ctr_for_position (CTR gap vs. own position-tier median) + visibility (log impressions_90d), no fitted weights",
    "scoreable_rows": int(len(scoreable)),
    "reason_code_counts": scoreable["reason_code"].value_counts().to_dict(),
    "base_rate_declined": round(float(base_rate), 4),
    "precision_at_10": round(float(precision_at_10), 4),
    "precision_at_10_labeled_n": int(labeled_n),
    "precision_at_10_note": "directional only, n below the 50-row floor",
}

with open("../outputs/baseline_metrics.json", "w") as f:
    _json.dump(metrics, f, indent=2)

print("Wrote work/outputs/baseline_metrics.json (commit this one -- the CSV stays out of git)")



WEAK PICKS (found by looking hard, per the skill's own instruction):
- #4: clicks are RISING (19 -> 36), not declining -- the rule flagged it purely on the
  static CTR-gap number and missed the page's own recent direction. A rule this simple
  can't see trend, only snapshot -- a real weakness worth naming, not hiding.
- #3, #7, #10: clicks_prev_30d is 4, 0, and 0 -- all below my own reliability floor
  (>=5) from ML-03. These three picks rest entirely on the CTR-gap belief with no
  click-based cross-check at all.

LEAKAGE CHECK:
- No FlyRank product flags used anywhere (health_score, priority_score, action_type,
  refresh_tier are not in this dataset at all, and were never referenced).
- Honest caveat: ctr, avg_position, and impressions_90d are 90-day aggregates that
  PARTIALLY OVERLAP the same last_30d window used to build my declined label (from
  ML-03). That means precision@10 below is a same-period consistency check -- does the
  rule agree with what already happened -- NOT a 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.